cell 1

In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
!pip install datasets av  # uncomment on first run

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 22.4 MB/s eta 0:00:00


cell 2

In [2]:
import sys
import random
import datetime
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "3Dconv"
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    print(f"Cloned → {REPO_DIR}")
else:
    print(f"Repo already present at {REPO_DIR}")

os.chdir(REPO_DIR)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

Cloned → /content/anyup
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


cell 3

In [19]:
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class TStage:
    t:           int
    start_step:  int
    batch_size:  int


@dataclass
class TrainConfig:
    # ── Experiment ────────────────────
    experiment_name: str = "run_02"

    # ── Paths ───────────────────────
    model_ckpt_2d:  Optional[str] = "checkpoints/anyup_2d.pth"
    checkpoint_dir: str           = "checkpoints/anyup3d"
    resume:         Optional[str] = None
    log_dir:        str           = "runs/anyup3d"
    dataset_root:   str           = "data/videos"

    # ── Teacher ───────────────────────
    encoder:       str = "videomae"
    encoder_model: str = "MCG-NJU/videomae-base"

    # ── Model ───────────────────────
    input_dim:       int   = 3
    qk_dim:          int   = 128    # ↓ reduce to 64 to save memory
    num_heads:       int   = 4      # must divide qk_dim evenly
    window_ratio:    float = 0.1    # ↓ reduce to cut attention cost; also affects memory
    kernel_size_lfu: int   = 5
    t_k:             int   = 3
    window_t:         int   = 4


    # ── Spatial resolution ───────────────────────
    # Relationship: img_size = crop_size // 2, source_size > crop_size
    source_size: int = 256   # frames loaded at this size; must be > crop_size
    crop_size:   int = 224   # spatial tube crop fed to VideoMAE for GT (teacher native)
                             #   ↑ if you change this, also update img_size below
    img_size:    int = 112   # AnyUp input (= crop_size // 2)
                             #   ↑ must be crop_size // 2 for 2× upsample to work

    # ── T-curriculum ───────────────────────
    t_schedule: List[TStage] = field(default_factory=lambda: [
        TStage(t=2,  start_step=0,     batch_size=4),   # ↓ reduce batch_size if OOM
        TStage(t=4,  start_step=5000,  batch_size=2),   # depends on t=2 batch above
        TStage(t=8,  start_step=15000, batch_size=1),   # depends on t=4 batch above
        # TStage(t=16, start_step=30000, batch_size=1), # uncomment for A100 only
    ])

    # ── Optimizer ───────────────────────
    lr:                 float = 2e-4
    weight_decay:       float = 1e-4
    grad_clip_max_norm: float = 1.0

    # ── Loss weights ───────────────────────
    lambda_cos_mse:               float = 1.0
    lambda_input_consistency:     float = 0.1
    lambda_self_consistency:      float = 0.1
    lambda_temporal_consistency:  float = 0.2
    temporal_lambda_warmup_steps: int   = 5000

    # ── Training duration ───────────────────────
    max_steps:          int = 50_000
    save_every_n_steps: int = 250
    val_every_n_steps:  int = 1000
    log_every_n_steps:  int = 50

    # ── Data ───────────────────────
    num_workers: int  = 2    # set to 0 if Colab throws Bus errors
    pin_memory:  bool = True

    # ── Misc ───────────────────────
    seed:        int  = 42
    debug:       bool = False
    debug_steps: int  = 10


cfg = TrainConfig()

# ============================================================
# COLAB OVERRIDES — edit this block for every new run.
# ============================================================
DRIVE_ROOT = "/content/drive/MyDrive/anyup3d"   # ← your Drive folder

cfg.experiment_name = "run_02"
cfg.model_ckpt_2d   = f"{DRIVE_ROOT}/anyup_2d.pth"
cfg.checkpoint_dir  = f"{DRIVE_ROOT}/checkpoints/{cfg.experiment_name}"
cfg.log_dir         = f"{DRIVE_ROOT}/runs/{cfg.experiment_name}"
cfg.dataset_root    = "/content/local_dataset"

# cfg.resume = f"{DRIVE_ROOT}/checkpoints/run_01/anyup3d_step46000.pth"
# cfg.debug  = True

print(cfg)

TrainConfig(experiment_name='run_02', model_ckpt_2d='/content/drive/MyDrive/anyup3d/anyup_2d.pth', checkpoint_dir='/content/drive/MyDrive/anyup3d/checkpoints/run_02', resume=None, log_dir='/content/drive/MyDrive/anyup3d/runs/run_02', dataset_root='/content/local_dataset', encoder='videomae', encoder_model='MCG-NJU/videomae-base', input_dim=3, qk_dim=128, num_heads=4, window_ratio=0.1, kernel_size_lfu=5, t_k=3, window_t=4, source_size=256, crop_size=224, img_size=112, t_schedule=[TStage(t=2, start_step=0, batch_size=4), TStage(t=4, start_step=5000, batch_size=2), TStage(t=8, start_step=15000, batch_size=1)], lr=0.0002, weight_decay=0.0001, grad_clip_max_norm=1.0, lambda_cos_mse=1.0, lambda_input_consistency=0.1, lambda_self_consistency=0.1, lambda_temporal_consistency=0.2, temporal_lambda_warmup_steps=5000, max_steps=50000, save_every_n_steps=250, val_every_n_steps=1000, log_every_n_steps=50, num_workers=2, pin_memory=True, seed=42, debug=False, debug_steps=10)


cell 4

In [6]:
import numpy as np

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda


cell 5

In [7]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted.")
except ModuleNotFoundError:
    print("Not in Colab — Drive mount skipped.")

timestamp   = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_log_dir = Path(cfg.log_dir) / timestamp
run_log_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(run_log_dir))

print(f"TensorBoard log dir : {run_log_dir}")
print("Launch with:  %load_ext tensorboard")
print(f"              %tensorboard --logdir {cfg.log_dir}")

Mounted at /content/drive
Drive mounted.
TensorBoard log dir : /content/drive/MyDrive/anyup3d/runs/run_02/2026-04-27_12-35-22
Launch with:  %load_ext tensorboard
              %tensorboard --logdir /content/drive/MyDrive/anyup3d/runs/run_02


cell 6

In [8]:
from transformers import VideoMAEModel

print(f"Loading teacher: {cfg.encoder} — {cfg.encoder_model} ...")

if cfg.encoder == "videomae":
    encoder      = VideoMAEModel.from_pretrained(cfg.encoder_model).to(device).eval()
    ENCODER_DIM  = encoder.config.hidden_size    # 768
    PATCH_SIZE   = encoder.config.patch_size     # 16
    TEACHER_SIZE = encoder.config.image_size     # 224
    TUBELET_SIZE = encoder.config.tubelet_size   # 2
else:
    raise ValueError(
        f"Unsupported encoder '{cfg.encoder}'. "
        f"Add a loading branch here before using a new encoder."
    )

for p in encoder.parameters():
    p.requires_grad_(False)

TOKEN_H     = cfg.crop_size // PATCH_SIZE    # 14
TOKEN_W     = cfg.crop_size // PATCH_SIZE    # 14
TOKEN_H_LOW = cfg.img_size  // PATCH_SIZE    # 7
TOKEN_W_LOW = cfg.img_size  // PATCH_SIZE    # 7

print(f"ENCODER_DIM : {ENCODER_DIM}")
print(f"GT tokens   : {TOKEN_H}×{TOKEN_W}   (from {cfg.crop_size}-px crop)")
print(f"Coarse tok. : {TOKEN_H_LOW}×{TOKEN_W_LOW}  (from {cfg.img_size}-px input)")

Loading teacher: videomae — MCG-NJU/videomae-base ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/377M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weig

ENCODER_DIM : 768
GT tokens   : 14×14   (from 224-px crop)
Coarse tok. : 7×7  (from 112-px input)


cell 7

In [10]:
from anyup.model import AnyUp
from scripts.load_2d_weights import build_3d_state_dict

anyup = AnyUp(
    input_dim=cfg.input_dim,
    qk_dim=cfg.qk_dim,
    num_heads=cfg.num_heads,
    window_ratio=cfg.window_ratio,
    kernel_size_lfu=cfg.kernel_size_lfu,
    t_k=cfg.t_k,
).to(device)

if cfg.model_ckpt_2d and Path(cfg.model_ckpt_2d).exists():
    print(f"Loading 2D checkpoint: {cfg.model_ckpt_2d}")
    raw = torch.load(cfg.model_ckpt_2d, map_location="cpu", weights_only=False)
    sd  = raw
    for key in ("anyup", "model", "state_dict", "weights", "params"):
        if isinstance(sd, dict) and key in sd:
            sd = sd[key]; break
    adapted_sd, _ = build_3d_state_dict(sd, anyup)
    anyup.load_state_dict(adapted_sd, strict=True)
    print(f"  Loaded {len(adapted_sd)} tensors with temporal centre-init")
elif cfg.model_ckpt_2d:
    print(f"[WARNING] 2D ckpt not found at '{cfg.model_ckpt_2d}' — random init")
else:
    print("Random init (no 2D checkpoint configured)")

global_step = 0
data_epoch  = 0

if cfg.resume and Path(cfg.resume).exists():
    print(f"Resuming from: {cfg.resume}")
    ckpt        = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    anyup.load_state_dict(ckpt["anyup"])
    global_step = ckpt.get("global_step", 0)
    data_epoch  = ckpt.get("data_epoch",  0)
    print(f"  Resumed at step {global_step}, data_epoch {data_epoch}")

scaler = torch.cuda.amp.GradScaler('cuda')
anyup.train()
print(f"Trainable params: {sum(p.numel() for p in anyup.parameters() if p.requires_grad):,}")

Loading 2D checkpoint: /content/drive/MyDrive/anyup3d/anyup_2d.pth
  Loaded 73 tensors with temporal centre-init
Trainable params: 1,540,736


/tmp/ipykernel_1005/1444115564.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler('cuda')


cell 8

In [11]:
from anyup.loss import Cosine_MSE

criterion = Cosine_MSE()


def cosine_mse_3d(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    B, C, T, H, W = pred.shape
    return criterion(pred.view(B, C, T * H, W), target.view(B, C, T * H, W))["total"]


def input_consistency_loss(
    pred:     torch.Tensor,
    guidance: torch.Tensor,
) -> torch.Tensor:
    B, C, T, H, W   = pred.shape
    _, _, _, Hg, Wg = guidance.shape

    pred_flat     = pred.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
    guidance_flat = guidance.permute(0, 2, 1, 3, 4).reshape(B * T, C, Hg, Wg)

    pred_pooled = F.adaptive_avg_pool2d(pred_flat, (Hg, Wg))

    pred_coarse = pred_pooled.view(B, T, C, Hg, Wg).permute(0, 2, 1, 3, 4)
    guid_coarse = guidance_flat.view(B, T, C, Hg, Wg).permute(0, 2, 1, 3, 4)

    return cosine_mse_3d(pred_coarse, guid_coarse)


def get_temporal_lambda(step: int) -> float:
    warmup = cfg.temporal_lambda_warmup_steps
    if warmup <= 0:
        return cfg.lambda_temporal_consistency
    return cfg.lambda_temporal_consistency * min(1.0, step / warmup)


print("Loss functions ready.")

Loss functions ready.


cell 9

In [12]:
import shutil
import subprocess
import pandas as pd
import torchvision.transforms.v2 as Tv2
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader, Dataset
import av

WORK_ROOT  = Path("/content/ucf101_top10")
TRAIN_DIR  = WORK_ROOT / "train"
TEST_DIR   = WORK_ROOT / "test"
TRAIN_CSV  = WORK_ROOT / "train.csv"
TEST_CSV   = WORK_ROOT / "test.csv"
DRIVE_TAR  = Path(DRIVE_ROOT) / "ucf101_top10.tar.gz"
TOP_N      = 10


def _run(cmd, **kw):
    subprocess.run(cmd, check=True, **kw)


def _prepare_top10_from_scratch(work_root: Path):
    work_root.mkdir(parents=True, exist_ok=True)
    os.chdir(work_root)
    rar_path = work_root / "UCF101.rar"
    zip_path = work_root / "UCF101TrainTestSplits-RecognitionTask.zip"
    data_dir = work_root / "data"
    splits   = work_root / "ucfTrainTestlist"

    if not rar_path.exists():
        print("Downloading UCF-101 (~6.5 GB) ...")
        _run(["wget", "-q", "--show-progress", "--no-check-certificate",
              "https://www.crcv.ucf.edu/data/UCF101/UCF101.rar", "-O", str(rar_path)])
    if not zip_path.exists():
        _run(["wget", "-q", "--no-check-certificate",
              "https://www.crcv.ucf.edu/data/UCF101/UCF101TrainTestSplits-RecognitionTask.zip",
              "-O", str(zip_path)])

    if not data_dir.exists() or not any(data_dir.glob("*.avi")):
        _run(["apt-get", "-q", "install", "-y", "unrar"])
        data_dir.mkdir(exist_ok=True)
        _run(["unrar", "e", "-o+", str(rar_path), str(data_dir) + "/"])
    if not splits.exists():
        _run(["unzip", "-qq", "-o", str(zip_path), "-d", str(work_root)])

    def _load_split(txt):
        with open(splits / txt) as f:
            lines = [l for l in f.read().split("\n") if l]
        df = pd.DataFrame({"video_name": lines})
        df["tag"]        = df["video_name"].apply(lambda p: p.split("/")[0])
        df["video_name"] = df["video_name"].apply(lambda p: p.split("/")[1].split(" ")[0])
        return df

    train = _load_split("trainlist01.txt")
    test  = _load_split("testlist01.txt")
    topN  = train["tag"].value_counts().nlargest(TOP_N).index.tolist()
    train = train[train["tag"].isin(topN)].reset_index(drop=True)
    test  = test[test["tag"].isin(topN)].reset_index(drop=True)
    print(f"Top-{TOP_N}: {topN} | train={len(train)}, test={len(test)}")

    def _move(df, out_dir):
        out_dir.mkdir(exist_ok=True)
        for name in df["video_name"]:
            src = data_dir / name
            if src.exists():
                shutil.copy2(src, out_dir / name)

    _move(train, WORK_ROOT / "train")
    _move(test,  WORK_ROOT / "test")
    train.to_csv(WORK_ROOT / "train.csv", index=False)
    test .to_csv(WORK_ROOT / "test.csv",  index=False)
    shutil.rmtree(data_dir, ignore_errors=True)
    rar_path.unlink(missing_ok=True)


need_prep = not (TRAIN_CSV.exists() and TRAIN_DIR.exists() and any(TRAIN_DIR.glob("*.avi")))
if need_prep:
    print("Downloading UCF-101 top-10 from scratch ...")
    _prepare_top10_from_scratch(WORK_ROOT)
else:
    print(f"Dataset ready at {WORK_ROOT}")

frame_tf = Tv2.Compose([
    Tv2.ToImage(),
    Tv2.Resize(cfg.source_size,
               interpolation=InterpolationMode.BILINEAR, antialias=True),
    Tv2.CenterCrop(cfg.source_size),
    Tv2.ToDtype(torch.float32, scale=True),
    Tv2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class UCF101TopNDataset(Dataset):
    def __init__(self, csv_path, video_dir, t=8):
        self.df           = pd.read_csv(csv_path).reset_index(drop=True)
        self.video_dir    = Path(video_dir)
        self.t            = t
        self.classes      = sorted(self.df["tag"].unique().tolist())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def _sample_frames(self, path):
        try:
            container = av.open(str(path))
            stream    = container.streams.video[0]
            total     = stream.frames or self.t * 4
            start     = random.randint(0, max(0, total - self.t))
            wanted    = set(range(start, start + self.t))
            frames    = []
            for fi, frm in enumerate(container.decode(video=0)):
                if fi in wanted:
                    frames.append(frame_tf(frm.to_image().convert("RGB")))
                if fi >= start + self.t:
                    break
            container.close()
        except Exception:
            frames = []
        while len(frames) < self.t:
            frames.append(frames[-1].clone() if frames
                          else torch.zeros(3, cfg.source_size, cfg.source_size))
        return frames[:self.t]

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        frames = self._sample_frames(self.video_dir / row["video_name"])
        return {
            "video": torch.stack(frames),
            "label": self.class_to_idx[row["tag"]],
            "tag":   row["tag"],
        }


def build_loader(t: int, batch_size: int, epoch: int = 0, split: str = "train") -> DataLoader:
    csv  = TRAIN_CSV if split == "train" else TEST_CSV
    vdir = TRAIN_DIR if split == "train" else TEST_DIR
    ds   = UCF101TopNDataset(csv_path=csv, video_dir=vdir, t=t)
    g    = torch.Generator(); g.manual_seed(cfg.seed + epoch)
    return DataLoader(
        ds, batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        generator=g, drop_last=(split == "train"),
    )

print("Dataset ready.")

Top-10: ['Punch', 'PlayingCello', 'ShavingBeard', 'CricketShot', 'PlayingGuitar', 'TennisSwing', 'Drumming', 'HorseRiding', 'PlayingDhol', 'BoxingPunchingBag'] | train=1171, test=459
Dataset ready.


cell 10

In [13]:
optimizer = torch.optim.AdamW(anyup.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.max_steps, eta_min=cfg.lr * 0.01)

if cfg.resume and Path(cfg.resume).exists():
    ckpt = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    if "optimizer" in ckpt: optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt: scheduler.load_state_dict(ckpt["scheduler"])

print("Optimiser and scheduler ready.")

Optimiser and scheduler ready.


cell 11

In [14]:
ckpt_dir = Path(cfg.checkpoint_dir)
ckpt_dir.mkdir(parents=True, exist_ok=True)


def save_checkpoint(step: int, data_epoch: int, tag: str = "") -> Path:
    name = f"anyup3d_step{step}{('_' + tag) if tag else ''}.pth"
    path = ckpt_dir / name
    torch.save({
        "anyup":       anyup.state_dict(),
        "optimizer":   optimizer.state_dict(),
        "scheduler":   scheduler.state_dict(),
        "global_step": step,
        "data_epoch":  data_epoch,
        "cfg":         vars(cfg),
    }, path)
    print(f"Checkpoint → {path}")
    return path


print(f"Checkpoints → {ckpt_dir}")

Checkpoints → /content/drive/MyDrive/anyup3d/checkpoints/run_02


cell 12

In [15]:
@torch.no_grad()
def extract_teacher_features(video: torch.Tensor) -> torch.Tensor:
    B, T, C, H, W = video.shape
    model_frames  = encoder.config.num_frames
    model_T_tok   = model_frames // TUBELET_SIZE

    if T < TUBELET_SIZE:
        reps  = (TUBELET_SIZE + T - 1) // T
        video = video.repeat(1, reps, 1, 1, 1)[:, :TUBELET_SIZE]
        T     = TUBELET_SIZE

    T_tok = T // TUBELET_SIZE

    pv = video.to(device)
    if H != TEACHER_SIZE or W != TEACHER_SIZE:
        pv = F.interpolate(
            pv.reshape(B * T, C, H, W),
            size=(TEACHER_SIZE, TEACHER_SIZE),
            mode="bilinear", align_corners=False,
        ).view(B, T, C, TEACHER_SIZE, TEACHER_SIZE)

    orig_pos_data = None
    if T_tok != model_T_tok:
        S        = TOKEN_H * TOKEN_W
        orig_pos = encoder.embeddings.position_embeddings.data
        p_interp = F.interpolate(
            orig_pos.view(1, model_T_tok, S, ENCODER_DIM).permute(0, 3, 1, 2).float(),
            size=(T_tok, S),
            mode="bilinear", align_corners=False,
        ).permute(0, 2, 3, 1).reshape(1, T_tok * S, ENCODER_DIM).to(orig_pos.dtype)
        orig_pos_data = orig_pos.clone()
        encoder.embeddings.position_embeddings.data = p_interp

    try:
        out = encoder(pixel_values=pv)
    finally:
        if orig_pos_data is not None:
            encoder.embeddings.position_embeddings.data = orig_pos_data

    return (
        out.last_hidden_state
        .reshape(B, T_tok, TOKEN_H, TOKEN_W, ENCODER_DIM)
        .permute(0, 4, 1, 2, 3)
        .contiguous()
    )


def sample_tube_crop(video: torch.Tensor):
    S  = cfg.source_size
    CH = cfg.crop_size
    IL = cfg.img_size

    y = random.randint(0, S - CH)
    x = random.randint(0, S - CH)

    crop_high = video[:, :, :, y : y + CH, x : x + CH].contiguous()

    B_v, T_v = crop_high.shape[:2]
    crop_low  = F.interpolate(
        crop_high.reshape(B_v * T_v, 3, CH, CH),
        size=(IL, IL),
        mode="bilinear", align_corners=False,
    ).view(B_v, T_v, 3, IL, IL)

    return crop_high, crop_low


@torch.no_grad()
def get_rgb_diff_gate(crop_low: torch.Tensor, T_tok: int) -> torch.Tensor:
    B, T, C, H, W = crop_low.shape

    token_frames = crop_low[:, ::TUBELET_SIZE]
    frame_diff   = (token_frames[:, 1:] - token_frames[:, :-1]).abs()
    frame_diff   = frame_diff.mean(dim=2, keepdim=True)

    Tt   = T_tok - 1
    gate = F.interpolate(
        frame_diff.reshape(B * Tt, 1, H, W),
        size=(TOKEN_H, TOKEN_W),
        mode="bilinear", align_corners=False,
    ).reshape(B, Tt, 1, TOKEN_H, TOKEN_W)

    gate = gate / (gate.amax(dim=(2, 3, 4), keepdim=True) + 1e-6)
    gate = 1.0 - gate

    return gate.permute(0, 2, 1, 3, 4)


def get_current_stage(step: int) -> TStage:
    stage = cfg.t_schedule[0]
    for s in cfg.t_schedule:
        if step >= s.start_step:
            stage = s
    return stage


print("Training utilities ready.")

Training utilities ready.


cell 13

In [18]:
max_steps     = cfg.debug_steps if cfg.debug else cfg.max_steps
current_stage = get_current_stage(global_step)

loader      = build_loader(current_stage.t, current_stage.batch_size, epoch=data_epoch)
loader_iter = iter(loader)

anyup.train()

pbar = tqdm(range(global_step, max_steps), initial=global_step, total=max_steps, desc="Training")

for step in pbar:

    new_stage = get_current_stage(step)
    if new_stage.t != current_stage.t or new_stage.batch_size != current_stage.batch_size:
        print(f"\n[Step {step}] Curriculum: T={current_stage.t}→{new_stage.t}, "
              f"BS={current_stage.batch_size}→{new_stage.batch_size}")
        current_stage = new_stage
        data_epoch   += 1
        loader        = build_loader(current_stage.t, current_stage.batch_size, epoch=data_epoch)
        loader_iter   = iter(loader)

    try:
        batch = next(loader_iter)
    except StopIteration:
        data_epoch   += 1
        loader        = build_loader(current_stage.t, current_stage.batch_size, epoch=data_epoch)
        loader_iter   = iter(loader)
        batch         = next(loader_iter)

    video = batch["video"].to(device)
    B, T, C, S, _ = video.shape

    with torch.no_grad():
        crop_high, crop_low = sample_tube_crop(video)

    with autocast(device_type=device.type, dtype=torch.bfloat16):

        gt_feats       = extract_teacher_features(crop_high)      # (B, 768, T_tok, 14, 14)
        guidance_feats = extract_teacher_features(crop_low)    # (B, 768, T_tok, 14, 14)

        # Downsample to 7×7 so AnyUp has something to actually upsample
        B_, C_, Tt_, H_, W_ = guidance_feats.shape
        guidance_feats_low = F.interpolate(
            guidance_feats.reshape(B_ * Tt_, C_, H_, W_),
            size=(TOKEN_H_LOW, TOKEN_W_LOW),                   # 7×7
            mode="bilinear", align_corners=False,
        ).reshape(B_, C_, Tt_, TOKEN_H_LOW, TOKEN_W_LOW)       # (B, 768, T_tok, 7, 7)

        T_tok    = gt_feats.shape[2]
        out_size = (TOKEN_H, TOKEN_W)

        # ── AnyUp3D forward ───────────────────────────────────────────────────
        crop_high_5d = crop_high.permute(0, 2, 1, 3, 4)          # (B, 3, T, 224, 224) — cross-attention guidance
        pred_feats   = anyup(crop_high_5d, guidance_feats_low, output_size=out_size)
        # guidance_feats: (B, 768, T_tok, 7, 7) — coarse features to upsample
        # crop_high_5d:   (B, 3,   T,     224, 224) — HR pixels for spatial guidance
        # pred_feats:     (B, 768, T_tok, 14, 14) — upsampled output                          # ← FIX

        loss_rec = cosine_mse_3d(pred_feats, gt_feats) * cfg.lambda_cos_mse

        loss_inp = torch.tensor(0.0, device=device)
        if cfg.lambda_input_consistency > 0:
            loss_inp = input_consistency_loss(pred_feats, guidance_feats_low) \
                       * cfg.lambda_input_consistency

        loss_aug = torch.tensor(0.0, device=device)
        if cfg.lambda_self_consistency > 0:
            aug_low    = crop_low + 0.05 * torch.randn_like(crop_low)
            with torch.no_grad():
                gt_aug = extract_teacher_features(aug_low)
            B_, C_, Tt_, H_, W_ = gt_aug.shape
            gt_aug_low = F.interpolate(
                gt_aug.reshape(B_ * Tt_, C_, H_, W_),
                size=(TOKEN_H_LOW, TOKEN_W_LOW),
                mode="bilinear", align_corners=False,
            ).reshape(B_, C_, Tt_, TOKEN_H_LOW, TOKEN_W_LOW)
            pred_aug = anyup(crop_high_5d, gt_aug_low, output_size=out_size)
            loss_aug   = cosine_mse_3d(pred_aug, pred_feats.detach()) \
                         * cfg.lambda_self_consistency

        loss_temp = torch.tensor(0.0, device=device)
        t_lam = get_temporal_lambda(step)
        if T_tok > 1 and t_lam > 0:
            feat_diff = pred_feats[:, :, 1:] - pred_feats[:, :, :-1]
            gate      = get_rgb_diff_gate(crop_low, T_tok)
            if gate.shape[-2:] != feat_diff.shape[-2:]:
                Tt   = T_tok - 1
                gate = F.interpolate(
                    gate.reshape(B * Tt, 1, *gate.shape[-2:]),
                    size=feat_diff.shape[-2:],
                    mode="bilinear", align_corners=False,
                ).reshape(B, 1, Tt, *feat_diff.shape[-2:])
            loss_temp = (feat_diff.pow(2) * gate).mean() * t_lam

        total_loss = loss_rec + loss_inp + loss_aug + loss_temp

    optimizer.zero_grad()
    scaler.scale(total_loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(anyup.parameters(), cfg.grad_clip_max_norm)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    if step % cfg.log_every_n_steps == 0:
        lr = optimizer.param_groups[0]["lr"]
        writer.add_scalar("loss/total",    total_loss.item(), step)
        writer.add_scalar("loss/rec",      loss_rec.item(),   step)
        writer.add_scalar("loss/inp",      loss_inp.item(),   step)
        writer.add_scalar("loss/aug",      loss_aug.item(),   step)
        writer.add_scalar("loss/temporal", loss_temp.item(),  step)
        writer.add_scalar("lr",            lr,                step)
        writer.add_scalar("curriculum/T",  current_stage.t,   step)
        writer.add_scalar("curriculum/bs", current_stage.batch_size, step)
        writer.add_scalar("data/epoch",    data_epoch,        step)
        pbar.set_postfix({
            "loss": f"{total_loss.item():.4f}",
            "rec":  f"{loss_rec.item():.4f}",
            "inp":  f"{loss_inp.item():.4f}",
            "T":    current_stage.t,
            "lr":   f"{lr:.2e}",
        })

    if (step + 1) % cfg.save_every_n_steps == 0:
        save_checkpoint(step + 1, data_epoch)

    if (step + 1) % cfg.val_every_n_steps == 0:
        anyup.eval()
        val_loader = build_loader(current_stage.t, current_stage.batch_size, split="test")
        val_losses = []
        with torch.no_grad():
            for val_batch in val_loader:
                val_video = val_batch["video"].to(device)
                val_high, val_low = sample_tube_crop(val_video)
                with autocast(device_type=device.type, dtype=torch.bfloat16):
                    val_gt  = extract_teacher_features(val_high)
                    val_gui = extract_teacher_features(val_low)
                    B_, C_, Tt_, H_, W_ = val_gui.shape
                    val_gui_low = F.interpolate(
                        val_gui.reshape(B_ * Tt_, C_, H_, W_),
                        size=(TOKEN_H_LOW, TOKEN_W_LOW),
                        mode="bilinear", align_corners=False,
                    ).reshape(B_, C_, Tt_, TOKEN_H_LOW, TOKEN_W_LOW)
                    val_pred = anyup(val_high.permute(0,2,1,3,4), val_gui_low, output_size=(TOKEN_H, TOKEN_W))
                    val_losses.append(cosine_mse_3d(val_pred, val_gt).item())
        writer.add_scalar("loss/val_rec", sum(val_losses) / len(val_losses), step + 1)
        anyup.train()

    if cfg.debug and step >= cfg.debug_steps - 1:
        print(f"Debug: stopping at step {step}.")
        break


save_checkpoint(max_steps, data_epoch, tag="final")
writer.flush(); writer.close()
print("Training complete.")

Training:   0%|          | 0/50000 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 196.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 53.81 MiB is free. Including non-PyTorch memory, this process has 14.51 GiB memory in use. Of the allocated memory 14.33 GiB is allocated by PyTorch, and 58.13 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

cell 14

In [ ]:
anyup.eval()
with torch.no_grad():
    dummy_video = torch.randn(1, 4, 3, cfg.source_size, cfg.source_size, device=device)
    dummy_high, dummy_low = sample_tube_crop(dummy_video)

    dummy_gt  = torch.randn(1, ENCODER_DIM, 2, TOKEN_H, TOKEN_W, device=device)
    dummy_gui = torch.randn(1, ENCODER_DIM, 2, TOKEN_H, TOKEN_W, device=device)

    dummy_low_5d  = dummy_low.permute(0, 2, 1, 3, 4)
    dummy_high_5d = dummy_high.permute(0, 2, 1, 3, 4)
    out = anyup(dummy_low_5d, dummy_high_5d, output_size=(TOKEN_H, TOKEN_W))

    assert out.shape == dummy_gt.shape, f"Shape mismatch: {out.shape} vs {dummy_gt.shape}"

    gate = get_rgb_diff_gate(dummy_low, T_tok=2)
    assert gate.shape == (1, 1, 1, TOKEN_H, TOKEN_W), f"Gate shape wrong: {gate.shape}"

    loss_inp_test = input_consistency_loss(out, dummy_gui)
    assert loss_inp_test.ndim == 0, "input_consistency_loss should return a scalar"

    print(f"source → crop_high : {dummy_high.shape}")
    print(f"source → crop_low  : {dummy_low.shape}")
    print(f"AnyUp  input       : {dummy_low_5d.shape}")
    print(f"AnyUp  output      : {out.shape}")
    print(f"RGB-diff gate      : {gate.shape}")
    print(f"L_inp (scalar)     : {loss_inp_test.item():.4f}")
    print("Smoke test passed ✓")

anyup.train()